# CorrDiff - Fase 10 - Relações Espaciais ERA5 × Radar

Associação raw, centralizada dentro do patch, contrastes de eventos e cross-offset.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/10_spatial_era5_radar')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## 1. Canais e estratos

In [ ]:
channels = pd.read_parquet(OUT/'input_channel_metadata.parquet')
strata = pd.read_parquet(OUT/'patch_strata_counts.parquet')
display(channels)
display(strata)


## 2. Raw vs centralizado dentro do patch

In [ ]:
pixel = pd.read_parquet(OUT/'pixel_spatial_associations.parquet')
base = pixel[pixel.smoothing_window_pixels.eq(1)]
for radar_field in ['ge_30','ge_40','ge_45']:
    t = base[base.radar_field.eq(radar_field)].pivot(index='predictor', columns='association_mode', values='pearson_r')
    t = t.reindex(t['within_patch_centered'].abs().sort_values(ascending=False).index)
    display(t.head(12))


## 3. Triagem de escalas locais

In [ ]:
for predictor in ['tcwv','r_500','t_850','t2m','delta_r_500_850','delta_t_500_850']:
    t = pixel[(pixel.predictor.eq(predictor)) & (pixel.radar_field.eq('ge_40')) & (pixel.association_mode.eq('within_patch_centered'))].sort_values('smoothing_half_width_km')
    plt.figure(figsize=(7,4))
    plt.plot(t.smoothing_half_width_km, t.pearson_r, marker='o')
    plt.xlabel('Raio da janela local (km)')
    plt.ylabel('Pearson r centralizado')
    plt.title(f'{predictor} × >=40 dBZ')
    plt.axhline(0)
    plt.tight_layout()
    plt.show()


## 4. Contraste evento menos background

In [ ]:
contrast = pd.read_parquet(OUT/'within_patch_event_contrasts.parquet')
for event_id in ['ge_30','ge_40','ge_45']:
    t = contrast[contrast.event_id.eq(event_id)].copy()
    t['abs_delta'] = t.mean_delta_in_patch_sigma.abs()
    display(t.sort_values('abs_delta', ascending=False).head(12)[['predictor','mean_event_minus_background','mean_delta_in_patch_sigma','weighted_fraction_positive_delta']])


## 5. Cross-offset

In [ ]:
offset = pd.read_parquet(OUT/'spatial_cross_offset_associations.parquet')
for predictor in ['tcwv','r_500','t_850','delta_r_500_850']:
    t = offset[(offset.predictor.eq(predictor)) & (offset.radar_field.eq('ge_40')) & (offset.association_mode.eq('within_patch_centered'))].copy()
    display(t.sort_values('pearson_r', ascending=False)[['offset_direction','offset_distance_km','pearson_r']])


## 6. Relação em nível de patch

In [ ]:
patch = pd.read_parquet(OUT/'patch_level_associations.parquet')
for metric in ['max_dbz','event_pixel_fraction_ge_30','event_pixel_fraction_ge_40','event_pixel_fraction_ge_45']:
    t = patch[patch.radar_field.eq(metric)].copy()
    t['abs_r'] = t.pearson_r.abs()
    display(t.sort_values('abs_r', ascending=False).head(12)[['predictor','pearson_r']])


## 7. Estrutura sazonal local

In [ ]:
season = pd.read_parquet(OUT/'pixel_spatial_associations_by_season.parquet')
t = season[(season.smoothing_window_pixels.eq(1)) & (season.association_mode.eq('within_patch_centered')) & (season.radar_field.eq('ge_45'))]
display(t.sort_values(['season_code','pearson_r'], ascending=[True,False]))


## Leitura científica

Priorize relações que persistem após centralização dentro do patch e que também aparecem no contraste evento-background. Associações fortes apenas no modo raw podem ser dominadas por diferenças de regime entre patches. A triagem por janelas não deve ser confundida com resolução física do ERA5; a Fase 11 fará a análise formal de escalas.